In [1]:
import sqlite3
import pandas as pd
import nflreadpy as nfl

print(f"SQLite version: {sqlite3.sqlite_version}")
print("All imports successful")

SQLite version: 3.53.0
All imports successful


In [2]:
# Pull 2024 play-by-play data
pbp_2024 = nfl.load_pbp([2024]).to_pandas()

# Select the most relevant columns for SQL practice
cols = ['game_id', 'week', 'posteam', 'defteam', 'down', 'ydstogo',
        'yardline_100', 'play_type', 'yards_gained', 'epa', 'wpa',
        'passer_player_name', 'rusher_player_name', 'receiver_player_name',
        'pass_location', 'air_yards', 'complete_pass', 'qb_hit', 'sack',
        'posteam_score', 'defteam_score', 'season']

pbp_slim = pbp_2024[cols].copy()

# Create a local SQLite database and load the data into it
conn = sqlite3.connect('nfl_2024.db')
pbp_slim.to_sql('plays', conn, if_exists='replace', index=False)

print(f"Database created successfully")
print(f"Total rows loaded: {len(pbp_slim)}")

Database created successfully
Total rows loaded: 49492


In [3]:
# Helper function to run SQL queries and return a clean dataframe
def query(sql):
    return pd.read_sql_query(sql, conn)

# Sanity check — look at the first 5 rows
query("""
    SELECT game_id, week, posteam, defteam, play_type, yards_gained, epa
    FROM plays
    LIMIT 5
""")

,game_id,week,posteam,defteam,play_type,yards_gained,epa
0,2024_01_ARI_BUF,1,NaN,NaN,NaN,NaN,0.000000
1,2024_01_ARI_BUF,1,ARI,BUF,kickoff,0.0,0.257819
2,2024_01_ARI_BUF,1,ARI,BUF,run,3.0,-0.200602
3,2024_01_ARI_BUF,1,ARI,BUF,pass,22.0,2.028874
4,2024_01_ARI_BUF,1,ARI,BUF,pass,9.0,0.754242


In [5]:

query("""
    SELECT 
        posteam AS team,
        COUNT(*) AS total_plays,
        SUM(yards_gained) AS total_yards,
        ROUND(AVG(yards_gained), 2) AS avg_yards_per_play
    FROM plays
    WHERE play_type IN ('pass', 'run')
    GROUP BY posteam
    ORDER BY total_yards DESC
    LIMIT 10
""")

,team,total_plays,total_yards,avg_yards_per_play
0,BAL,1167,8097.0,6.94
1,PHI,1341,7725.0,5.76
2,DET,1164,7504.0,6.45
3,WAS,1308,7496.0,5.73
4,BUF,1202,7251.0,6.03
5,TB,1128,7091.0,6.29
6,GB,1081,6623.0,6.13
7,KC,1228,6474.0,5.27
8,SF,1014,6405.0,6.32
9,LA,1147,6346.0,5.53


In [6]:
query("""
    SELECT 
        posteam AS team,
        COUNT(*) AS total_plays,
        SUM(yards_gained) AS total_yards,
        ROUND(AVG(yards_gained), 2) AS avg_yards_per_play
    FROM plays
    WHERE play_type IN ('pass', 'run')
    GROUP BY posteam
    ORDER BY total_yards DESC
""")

,team,total_plays,total_yards,avg_yards_per_play
0,BAL,1167,8097.0,6.94
1,PHI,1341,7725.0,5.76
2,DET,1164,7504.0,6.45
3,WAS,1308,7496.0,5.73
4,BUF,1202,7251.0,6.03
5,TB,1128,7091.0,6.29
6,GB,1081,6623.0,6.13
7,KC,1228,6474.0,5.27
8,SF,1014,6405.0,6.32
9,LA,1147,6346.0,5.53


In [7]:
query("""
    SELECT
        posteam AS team,
        COUNT(*) AS total_plays,
        SUM(CASE WHEN play_type = 'pass' THEN 1 ELSE 0 END) AS pass_plays,
        SUM(CASE WHEN play_type = 'run' THEN 1 ELSE 0 END) AS run_plays,
        ROUND(100.0 * SUM(CASE WHEN play_type = 'pass' THEN 1 ELSE 0 END) / COUNT(*), 1) AS pass_rate
    FROM plays
    WHERE play_type IN ('pass', 'run')
    GROUP BY posteam
    ORDER BY pass_rate DESC
    LIMIT 10
""")

,team,total_plays,pass_plays,run_plays,pass_rate
0,CIN,1069,701,368,65.6
1,CLE,1121,734,387,65.5
2,LV,1054,683,371,64.8
3,NYJ,1014,653,361,64.4
4,SEA,1018,648,370,63.7
5,DAL,1095,675,420,61.6
6,NYG,1051,640,411,60.9
7,CHI,1059,640,419,60.4
8,KC,1228,738,490,60.1
9,CAR,987,585,402,59.3


In [8]:
query("""
    SELECT
        p.posteam AS team,
        ROUND(AVG(CASE WHEN p.play_type = 'pass' THEN p.epa END), 3) AS avg_pass_epa,
        ROUND(AVG(CASE WHEN p.play_type = 'run' THEN p.epa END), 3) AS avg_run_epa,
        ROUND(AVG(p.epa), 3) AS avg_overall_epa
    FROM plays p
    WHERE p.play_type IN ('pass', 'run')
    GROUP BY p.posteam
    ORDER BY avg_overall_epa DESC
    LIMIT 10
""")

,team,avg_pass_epa,avg_run_epa,avg_overall_epa
0,BAL,0.321,0.129,0.219
1,BUF,0.248,0.127,0.190
2,DET,0.247,0.069,0.165
3,WAS,0.140,0.123,0.132
4,TB,0.192,0.050,0.129
5,PHI,0.133,0.107,0.119
6,CIN,0.143,-0.011,0.090
7,GB,0.117,0.025,0.071
8,SF,0.114,0.012,0.069
9,ARI,0.076,0.056,0.067


In [9]:
# Create a team info table to join against
team_info = pd.DataFrame({
    'team': ['CHI', 'GB', 'MIN', 'DET', 'DAL', 'PHI', 'NYG', 'WAS',
             'SF', 'LAR', 'SEA', 'ARI', 'BAL', 'CLE', 'PIT', 'CIN',
             'BUF', 'MIA', 'NE', 'NYJ', 'KC', 'LV', 'LAC', 'DEN',
             'HOU', 'IND', 'TEN', 'JAX', 'ATL', 'NO', 'TB', 'CAR'],
    'conference': ['NFC','NFC','NFC','NFC','NFC','NFC','NFC','NFC',
                   'NFC','NFC','NFC','NFC','AFC','AFC','AFC','AFC',
                   'AFC','AFC','AFC','AFC','AFC','AFC','AFC','AFC',
                   'AFC','AFC','AFC','AFC','NFC','NFC','NFC','NFC'],
    'division': ['NFC North','NFC North','NFC North','NFC North',
                 'NFC East','NFC East','NFC East','NFC East',
                 'NFC West','NFC West','NFC West','NFC West',
                 'AFC North','AFC North','AFC North','AFC North',
                 'AFC East','AFC East','AFC East','AFC East',
                 'AFC West','AFC West','AFC West','AFC West',
                 'AFC South','AFC South','AFC South','AFC South',
                 'NFC South','NFC South','NFC South','NFC South']
})

team_info.to_sql('teams', conn, if_exists='replace', index=False)
print("Teams table created successfully")

Teams table created successfully


In [10]:
query("""
    SELECT
        t.division,
        ROUND(AVG(p.epa), 3) AS avg_epa,
        ROUND(AVG(CASE WHEN p.play_type = 'pass' THEN p.epa END), 3) AS avg_pass_epa,
        ROUND(AVG(CASE WHEN p.play_type = 'run' THEN p.epa END), 3) AS avg_run_epa
    FROM plays p
    JOIN teams t ON p.posteam = t.team
    WHERE p.play_type IN ('pass', 'run')
    GROUP BY t.division
    ORDER BY avg_epa DESC
""")

,division,avg_epa,avg_pass_epa,avg_run_epa
0,NFC North,0.039,0.069,0.001
1,NFC West,0.035,0.055,0.008
2,AFC North,0.026,0.039,0.008
3,NFC East,0.021,0.004,0.041
4,AFC East,0.020,0.058,-0.031
5,NFC South,0.017,0.013,0.022
6,AFC West,-0.011,0.018,-0.053
7,AFC South,-0.059,-0.060,-0.058


In [11]:
query("""
    WITH bears_plays AS (
        SELECT
            week,
            play_type,
            yards_gained,
            epa,
            sack,
            down,
            ydstogo
        FROM plays
        WHERE posteam = 'CHI'
        AND play_type IN ('pass', 'run')
    ),
    weekly_summary AS (
        SELECT
            week,
            COUNT(*) AS total_plays,
            ROUND(AVG(epa), 3) AS avg_epa,
            SUM(yards_gained) AS total_yards,
            SUM(sack) AS total_sacks
        FROM bears_plays
        GROUP BY week
    )
    SELECT
        week,
        total_plays,
        avg_epa,
        total_yards,
        total_sacks
    FROM weekly_summary
    ORDER BY week
""")

,week,total_plays,avg_epa,total_yards,total_sacks
0,1,52,-0.249,152.0,2.0
1,2,66,-0.314,205.0,7.0
2,3,84,-0.202,395.0,4.0
3,4,52,0.083,266.0,3.0
4,5,67,0.214,428.0,1.0
5,6,58,0.293,376.0,3.0
6,8,61,-0.129,309.0,3.0
7,9,69,-0.228,242.0,6.0
8,10,59,-0.402,142.0,9.0
9,11,69,0.200,391.0,3.0


In [12]:
query("""
    WITH weekly_epa AS (
        SELECT
            week,
            ROUND(AVG(epa), 3) AS avg_epa
        FROM plays
        WHERE posteam = 'CHI'
        AND play_type IN ('pass', 'run')
        GROUP BY week
    )
    SELECT
        week,
        avg_epa,
        ROUND(AVG(avg_epa) OVER (
            ORDER BY week
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ), 3) AS rolling_3wk_avg,
        RANK() OVER (ORDER BY avg_epa DESC) AS epa_rank,
        ROUND(avg_epa - LAG(avg_epa) OVER (ORDER BY week), 3) AS change_from_prev_week
    FROM weekly_epa
    ORDER BY week
""")

,week,avg_epa,rolling_3wk_avg,epa_rank,change_from_prev_week
0,1,-0.249,-0.249,15,NaN
1,2,-0.314,-0.281,16,-0.065
2,3,-0.202,-0.255,11,0.112
3,4,0.083,-0.144,4,0.285
4,5,0.214,0.032,2,0.131
5,6,0.293,0.197,1,0.079
6,8,-0.129,0.126,8,-0.422
7,9,-0.228,-0.021,14,-0.099
8,10,-0.402,-0.253,17,-0.174
9,11,0.200,-0.143,3,0.602


In [14]:
query("""
    WITH team_weekly AS (
        SELECT
            p.posteam AS team,
            t.division,
            p.week,
            ROUND(AVG(p.epa), 3) AS avg_epa,
            COUNT(*) AS total_plays,
            SUM(p.sack) AS total_sacks
        FROM plays p
        JOIN teams t ON p.posteam = t.team
        WHERE p.play_type IN ('pass', 'run')
        GROUP BY p.posteam, t.division, p.week
    ),
    division_weekly AS (
        SELECT
            division,
            week,
            ROUND(AVG(avg_epa), 3) AS div_avg_epa
        FROM team_weekly
        GROUP BY division, week
    ),
    final AS (
        SELECT
            tw.team,
            tw.division,
            tw.week,
            tw.avg_epa,
            tw.total_sacks,
            dw.div_avg_epa,
            ROUND(tw.avg_epa - dw.div_avg_epa, 3) AS vs_division_avg,
            RANK() OVER (
                PARTITION BY tw.week
                ORDER BY tw.avg_epa DESC
            ) AS weekly_rank
        FROM team_weekly tw
        JOIN division_weekly dw
            ON tw.division = dw.division
            AND tw.week = dw.week
    )
    SELECT *
    FROM final
    WHERE team = 'CHI'
    ORDER BY week
""")

,team,division,week,avg_epa,total_sacks,div_avg_epa,vs_division_avg,weekly_rank
0,CHI,NFC North,1,-0.249,2.0,0.049,-0.298,24
1,CHI,NFC North,2,-0.314,7.0,-0.107,-0.207,30
2,CHI,NFC North,3,-0.202,4.0,0.017,-0.219,21
3,CHI,NFC North,4,0.083,3.0,0.113,-0.030,11
4,CHI,NFC North,5,0.214,1.0,-0.014,0.228,6
5,CHI,NFC North,6,0.293,3.0,0.318,-0.025,3
6,CHI,NFC North,8,-0.129,3.0,0.028,-0.157,27
7,CHI,NFC North,9,-0.228,6.0,-0.040,-0.188,25
8,CHI,NFC North,10,-0.402,9.0,-0.175,-0.227,26
9,CHI,NFC North,11,0.200,3.0,0.245,-0.045,7


In [16]:
query("""
    SELECT 
        passer_player_name AS quarterback,
        COUNT(*) AS total_attempts,
        SUM(sack) AS total_sacks
    FROM plays
    WHERE play_type = 'pass'
    GROUP BY passer_player_name
    ORDER BY total_sacks DESC
    LIMIT 5
""")

,quarterback,total_attempts,total_sacks
0,C.Williams,636,68.0
1,C.Stroud,655,63.0
2,S.Darnold,643,57.0
3,J.Hurts,503,51.0
4,J.Daniels,648,51.0


In [18]:
query("""
    SELECT
        posteam AS team,
        COUNT(*) AS total_plays,
        ROUND(AVG(epa), 3) AS avg_epa_per_play
    FROM plays
    WHERE play_type IN ('pass', 'run')
        AND yardline_100 <= 20
    GROUP BY posteam
    ORDER BY avg_epa_per_play DESC
""")

,team,total_plays,avg_epa_per_play
0,BAL,178,0.328
1,BUF,232,0.255
2,TB,175,0.211
3,WAS,255,0.208
4,DET,239,0.159
5,DEN,157,0.141
6,CIN,209,0.103
7,CAR,159,0.095
8,GB,195,0.091
9,KC,227,0.089


In [23]:
query("""
    WITH bears_plays AS (
        SELECT
            epa,
            down
        FROM plays
        WHERE posteam = 'CHI'
        AND play_type IN ('pass', 'run')
    ),
    down_summary AS (
        SELECT
            down,
            COUNT(*) AS total_plays,
            ROUND(AVG(epa), 3) AS avg_epa
        FROM bears_plays
        WHERE down IS NOT NULL
        GROUP BY down
    )
    SELECT
        down,
        total_plays,
        avg_epa
    FROM down_summary
    ORDER BY avg_epa DESC
""")

,down,total_plays,avg_epa
0,4.0,38,0.463
1,2.0,353,-0.017
2,1.0,430,-0.068
3,3.0,228,-0.289


In [27]:
query("""
    WITH team_epa AS (
        SELECT
            posteam AS team,
            ROUND(AVG(epa), 3) AS avg_epa
        FROM plays
        WHERE play_type IN ('pass', 'run')
        GROUP BY posteam
    )
    SELECT
        team,
        avg_epa,
        RANK() OVER (ORDER BY avg_epa DESC) AS epa_rank
    FROM team_epa
    ORDER BY epa_rank
""")

,team,avg_epa,epa_rank
0,BAL,0.219,1
1,BUF,0.190,2
2,DET,0.165,3
3,WAS,0.132,4
4,TB,0.129,5
5,PHI,0.119,6
6,CIN,0.090,7
7,GB,0.071,8
8,SF,0.069,9
9,ARI,0.067,10


In [29]:
query("""
    WITH third_down_plays AS (
        SELECT
            t.division AS division,
            p.epa AS epa
        FROM plays p
        JOIN teams t ON p.posteam = t.team
        WHERE p.play_type IN ('pass', 'run')
        AND p.down = 3
    )
    SELECT
        division,
        COUNT(*) AS total_plays,
        ROUND(AVG(epa), 3) AS avg_epa
    FROM third_down_plays
    GROUP BY division
    ORDER BY avg_epa DESC
""")

,division,total_plays,avg_epa
0,AFC North,903,0.033
1,NFC East,997,0.002
2,NFC West,596,-0.017
3,NFC North,861,-0.027
4,AFC West,942,-0.035
5,NFC South,859,-0.042
6,AFC East,871,-0.054
7,AFC South,898,-0.083
